# D4-equivariant CNN for Weak Lensing Shape Measurement

This notebook provides a minimal end-to-end example of the D4-equivariant CNN pipeline:

1. **Training example**: train a `Forward8_fixW_CNN` on an (image, ellipticity) dataset for 50 epochs.
2. **Inference example**: load a trained model and predict the ellipticity (e1, e2) of a simulated galaxy image.
3. **Pixel response & shear response**: compute the model's pixel-wise gradient and the shear response
   matrix used for unbiased calibration (metadetection-style).

### Background

In weak lensing, we estimate the shear $\mathbf{g}$ from the measured galaxy shapes $\mathbf{e}$.
A CNN shape estimator $f$ maps a postage-stamp image $I$ to a shape $(e_1, e_2) = f(I)$.
Because the estimator is differentiable, we can compute both:

- the **pixel response** $\partial e_i / \partial I$ (gradient of the model output w.r.t. the input image), and
- the **shear response** $R_{ij} = \partial e_i / \partial g_j$, obtained by chaining the model gradient
  with the analytic pixel response $\partial I / \partial g$ from FPFS.

These quantities enter the calibration of the multiplicative bias $m$ and additive bias $c$
(see `shape_measurement.py` and `cal_biases.py`).

Dependencies: `torch`, `numpy`, `matplotlib`, `galsim`, `anacal`, `pandas`, plus the `src/` package
in this repository.


In [ ]:
# ---------------------------------------------------------------------------
# Imports and device setup
# ---------------------------------------------------------------------------
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Galaxy image simulator (galaxy + PSF + optional noise)
from src.datasets.single_gal_sims import sim_simple_gal
# D4-equivariant CNN shape estimator: input [B,1,H,W] -> output [B,2] (e1,e2)
from src.architecture.forward8_CNN import Forward8_fixW_CNN
# Inference helper (takes [H,W] or [B,1,H,W], returns predicted [B,2])
from src.architecture.CNN_toolkit import predictor, shape_pixel_gradients
# FPFS pixel response (computes dI/dg from image + PSF)
from src.anacal.pixel_response import anacal_pix_r
# Training loop: train_model(model, images_path, csv_path, ...)
from src.training import train_model

# Use GPU if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")


## 1. Training Example

`train_model` reads images from `images_path` (`.npy`, shape `(N, H, W)`) and labels from `csv_path`
(the CSV must contain `split`, `id`, `e1`, `e2` columns).

The dataset loader (`src/datasets/single_gal_dataset.py`) implements spin-2–correct data augmentation:
90° rotations and reflections of the galaxy image are paired with the corresponding rotation/reflection
of the ellipticity label. This is essential for training an equivariant shape estimator.

This example uses the small test dataset shipped with the repository in `datasets/`
(`example_images.npy` + `example_catalog.csv`, 12k galaxies split into train/val/test).
Set `images_path` / `csv_path` below to your own dataset to train on a larger sample.


In [ ]:
# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
# ---- Data paths (use the bundled example dataset, or point to your own) ----
# images_path: .npy file, shape (N, H, W), float32, pixel values of the postage stamps
# csv_path   : .csv file, must contain the columns: id, split, e1, e2
#              - id    : unique integer per galaxy (used to index the .npy)
#              - split : 'train' / 'val' / 'test'
#              - e1,e2 : ground-truth ellipticity components (spin-2)
images_path = "./datasets/example_images.npy"
csv_path    = "./datasets/example_catalog.csv"

# Build the D4-equivariant CNN shape estimator.
# - num_layers    : number of convolution blocks (each block = Conv + GeLU + residual)
# - base_channels : number of channels in the first convolution layer
# - res_factor    : strength of the residual connection (0.1 in this example)
model = Forward8_fixW_CNN(num_layers=5, base_channels=32, res_factor=0.1).to(device)

# Run the training loop.
# - target   : "e" trains on ellipticity labels, "g" on shear labels
# - epochs   : number of full passes over the training set
# - batch_size / num_workers : dataloader settings
# - lr / wd  : AdamW learning rate and weight decay
# - pt_path  : where the best (lowest val-loss) weights are saved during training
model = train_model(
    model,
    images_path=images_path,
    csv_path=csv_path,
    target="e",          # or "g" to train on shear labels
    epochs=50,
    batch_size=500,
    num_workers=4,
    lr=1e-3,
    wd=1e-4,
    pt_path="./models/example_trained.pth",   # best-epoch weights are saved here
)

# Save the final weights (last epoch) separately
torch.save(model.state_dict(), "./models/example_trained_final.pth")
print("saved final weights to ./models/example_trained_final.pth")


## 2. Load Model & Single-Galaxy Inference

After training (or by directly using the pretrained weights in `models/`), we take a galaxy image
from the bundled example dataset and predict its ellipticity (e1, e2) with the model.

For the pixel-response demo below we also need a PSF image. `sim_simple_gal(..., return_psf=True)`
returns a stacked array `(2, H, W)` where index 0 is the galaxy image and index 1 is the PSF image.


In [ ]:
# ---------------------------------------------------------------------------
# Load model weights
# ---------------------------------------------------------------------------
# Prefer the just-trained checkpoint; if it does not exist (e.g. first run,
# or the training cell was skipped), fall back to a pretrained weight that is
# shipped with the repository in ./models/.
ckpt = "./models/example_trained_final.pth"
if not os.path.exists(ckpt):
    ckpt = "./models/forward8_CNN_nada_50ep.pth"
print(f"load checkpoint: {ckpt}")

# Rebuild the same architecture that was used for training. The architecture
# must match the checkpoint exactly (same num_layers/base_channels/res_factor),
# otherwise load_state_dict will raise a shape-mismatch error.
model2 = Forward8_fixW_CNN(num_layers=5, base_channels=32, res_factor=0.1).to(device)
model2.load_state_dict(torch.load(ckpt, map_location="cpu"))
model2.eval()   # switch to inference mode (disables dropout / batch-norm updates)


In [ ]:
# ---------------------------------------------------------------------------
# Load a real galaxy from the example dataset and predict its ellipticity
# ---------------------------------------------------------------------------
# The dataset images are real (noiseless) Sérsic galaxies with known PSF, so we
# can directly measure how well the model recovers the ground-truth ellipticity.
import pandas as pd

# Load the catalog and pick one galaxy from the test split
catalog = pd.read_csv(csv_path)
test_ids = catalog[catalog["split"] == "test"]["id"].values
gal_id = int(test_ids[0])
row = catalog[catalog["id"] == gal_id].iloc[0]

# Load the corresponding image from the .npy array
images_all = np.load(images_path, mmap_mode="r")
img = np.array(images_all[gal_id], dtype=np.float32)   # [H,W] single galaxy
e1_true, e2_true = float(row["e1"]), float(row["e2"])
print(f"test galaxy id={gal_id}, image shape: {img.shape}")
print(f"truth e1,e2 = ({e1_true:.3f}, {e2_true:.3f})")

# Predict the ellipticity with the loaded model
pred = predictor(model2, img, normalize="none")
print(f"pred  e1,e2 = ({pred[0]:.3f}, {pred[1]:.3f})")

# Visualize the input image and overlay the prediction in the title
plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray", origin="lower")
plt.title(f"pred e=({pred[0]:.2f}, {pred[1]:.2f})")
plt.axis("off")
plt.show()


## 3. Pixel Response & Shear Response (Advanced)

The model is differentiable pixel-by-pixel. Two response quantities are useful for calibration:

1. **Pixel response** $\partial e_i / \partial I$ — how the model output changes when a single pixel changes.
   Computed with `shape_pixel_gradients` (backprop through the network w.r.t. the input image).

2. **Shear response** $R_{ij} = \partial e_i / \partial g_j$ — how the measured shape responds to a true shear.
   By the chain rule:
   $$
   R_{ij} = \sum_{\text{pixels}} \frac{\partial e_i}{\partial I} \cdot \frac{\partial I}{\partial g_j},
   $$
   where $\partial I / \partial g_j$ is the **FPFS pixel response** computed by `anacal_pix_r`.
   The response matrix $R$ enters the calibration of the multiplicative bias:
   $$
   \langle e_i \rangle = c_i + R_{ij} g_j .
   $$
   (see `shape_measurement.py` and `cal_biases.py`).


In [ ]:
# ---------------------------------------------------------------------------
# 3a. Pixel response: gradient of the model output w.r.t. the input image
# ---------------------------------------------------------------------------
# shape_pixel_gradients(model, img, normalize="none") returns:
#   pred    : (B, 2) predicted shapes
#   grad_e1 : (B, 1, H, W)  d(e1)/dI  -- pixel map of the e1 gradient
#   grad_e2 : (B, 1, H, W)  d(e2)/dI  -- pixel map of the e2 gradient
pred, grad_e1, grad_e2 = shape_pixel_gradients(model2, img, normalize="none")

print("pred (e1, e2):", pred)

# Visualize the input image and the two pixel-response maps side by side.
# The gradient maps show which pixels drive the shape estimate most strongly.
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
vmax = max(np.abs(grad_e1).max(), np.abs(grad_e2).max())   # shared color scale
axs[0].imshow(img, cmap="gray", origin="lower")
axs[0].set_title("input")
im1 = axs[1].imshow(grad_e1[0, 0], cmap="bwr", origin="lower", vmin=-vmax, vmax=vmax)
axs[1].set_title(r"$\partial e_1/\partial I$")
im2 = axs[2].imshow(grad_e2[0, 0], cmap="bwr", origin="lower", vmin=-vmax, vmax=vmax)
axs[2].set_title(r"$\partial e_2/\partial I$")
plt.colorbar(im1, ax=axs[1]); plt.colorbar(im2, ax=axs[2])
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 4. Shear Response Matrix (Calibration)

We now combine the model's pixel response with the **FPFS pixel response** to obtain the shear
response matrix $R$ for a single galaxy:
$$
R_{ij} = \sum_{\text{pixels}} \frac{\partial e_i}{\partial I} \cdot \frac{\partial I}{\partial g_j}.
$$

`anacal_pix_r(img, psf)` returns a stacked `(5, H, W)` array
`[v, g1, g2, j1, j2]`, where `g1 = dI/dg1` and `g2 = dI/dg2` are the pixel-level shear responses.

> The example dataset stores only galaxy images, while AnaCal needs the PSF to build the pixel
> response. For this demo we regenerate a noiseless simulated galaxy *with* its PSF; the same code
> applies to any (image, psf) pair from your own pipeline.


In [ ]:
# ---------------------------------------------------------------------------
# 4a. Regenerate a simulated galaxy + PSF for the response demo
# ---------------------------------------------------------------------------
# sim_simple_gal(..., return_psf=True) returns a stacked (2, H, W) array:
#   [0] = galaxy image, [1] = PSF image.
img_psf = sim_simple_gal(
    e1=0.3, e2=-0.2,
    gal_flux=1000,
    gal_half_light_radius=0.7,
    sersic_n=1,
    psf_model="Moffat",
    psf_para=(3.5, 0.8),
    pixel_scale=0.2,
    image_size=64,
    noise_std=0,
    seed=42,
    return_psf=True,
    show=False,
)
img_r = img_psf[0].astype(np.float32)   # galaxy image [H,W]
psf_r = img_psf[1].astype(np.float32)   # PSF image    [H,W]

# Recompute the model's pixel response on this new image
pred_r, grad_e1_r, grad_e2_r = shape_pixel_gradients(model2, img_r, normalize="none")
print("pred (e1, e2) on simulated galaxy:", pred_r)

# ---------------------------------------------------------------------------
# 4b. FPFS pixel response (dI/dg1, dI/dg2) from the image and PSF
# ---------------------------------------------------------------------------
# anacal_pix_r(img, psf, ...) returns a stacked (5, H, W) float64 array:
#   q[0] = v   : resmoothed (weighted) image
#   q[1] = g1  : dI/dg1  (pixel response to shear g1)
#   q[2] = g2  : dI/dg2  (pixel response to shear g2)
#   q[3], q[4] : auxiliary first-moment fields (used internally for centering)
# Optional arguments:
#   sigma_arcsec      : Gaussian smoothing scale for the weight function
#   scale_arcsec_per_pix : pixel scale (0.2" for this example)
#   freq_lim          : frequency limit in Fourier space (arcsec^-1)
q = anacal_pix_r(
    img_r.astype(np.float64),   # galaxy image
    psf_r.astype(np.float64),   # PSF image
    sigma_arcsec=0.85 / 2.355,
    scale_arcsec_per_pix=0.2,
    freq_lim=10.0,
)
print("q array shape:", q.shape)          # expect (5, H, W)
print("q fields: v, g1, g2, j1, j2")

# Visualize the two FPFS pixel responses dI/dg1 and dI/dg2
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
vmax = max(np.abs(q[1]).max(), np.abs(q[2]).max())
axs[0].imshow(img_r, cmap="gray", origin="lower"); axs[0].set_title("input image")
im1 = axs[1].imshow(q[1], cmap="bwr", origin="lower", vmin=-vmax, vmax=vmax)
axs[1].set_title(r"$\partial I/\partial g_1$")
im2 = axs[2].imshow(q[2], cmap="bwr", origin="lower", vmin=-vmax, vmax=vmax)
axs[2].set_title(r"$\partial I/\partial g_2$")
plt.colorbar(im1, ax=axs[1]); plt.colorbar(im2, ax=axs[2])
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 4c. Shear response matrix R[i,j] = sum_pixels (de_i/dI) * (dI/dg_j)
# ---------------------------------------------------------------------------
# grad_e1_r/grad_e2_r are the model pixel responses (B,1,H,W); q[1], q[2] are the
# FPFS pixel responses dI/dg1, dI/dg2. Contracting them gives the 2x2 response
# matrix for this single galaxy.
R = np.zeros((2, 2))
R[0, 0] = (grad_e1_r[0, 0] * q[1]).sum()    # de1/dg1
R[0, 1] = (grad_e1_r[0, 0] * q[2]).sum()    # de1/dg2
R[1, 0] = (grad_e2_r[0, 0] * q[1]).sum()    # de2/dg1
R[1, 1] = (grad_e2_r[0, 0] * q[2]).sum()    # de2/dg2

print("single-galaxy shear response R:")
print(R)

# Sanity check: the diagonal terms (R00, R11) are the response of e1 to g1 and
# e2 to g2; the off-diagonal terms should be small for a well-behaved estimator.
print(f"R00 (de1/dg1) = {R[0,0]:.3f}")
print(f"R11 (de2/dg2) = {R[1,1]:.3f}")

# Calibrated shear (single-galaxy demo): g = R^{-1} * e
try:
    g_cal = np.linalg.solve(R, pred_r)
    print("calibrated shear (single galaxy):", g_cal)
except np.linalg.LinAlgError:
    print("R is singular; need an ensemble average for a stable inversion.")
